# Data scraping: food image dataset

Scrapes Bing Images to build an `ImageFolder`-style dataset (`<root>/<class>/*.jpg`) for PyTorch.
Run in Google Colab. Scraped images are copyrighted by their owners — use for personal study only.

In [ ]:
!pip install -q playwright pillow requests imagehash
!playwright install --with-deps chromium > /dev/null

In [ ]:
import os, uuid, requests, imagehash
from io import BytesIO
from PIL import Image
from urllib.parse import quote_plus
from playwright.async_api import async_playwright

async def collect_urls(query, limit, max_scrolls=30):
    """Scroll Bing Images until we have enough candidate URLs (2x limit, since many downloads fail)."""
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox", "--disable-dev-shm-usage"])
        page = await browser.new_page(locale="en-US")
        await page.goto(f"https://www.bing.com/images/search?q={quote_plus(query)}",
                        wait_until="domcontentloaded", timeout=60000)
        urls = []
        for _ in range(max_scrolls):
            await page.mouse.wheel(0, 1500)
            await page.wait_for_timeout(1500)
            # Click "See more images" if Bing shows it
            more = page.locator("a.btn_seemore")
            if await more.count() and await more.first.is_visible():
                await more.first.click()
            urls = await page.locator("a.iusc").evaluate_all("""
                els => els.map(e => { try { return JSON.parse(e.getAttribute('m')).murl; }
                                      catch { return null; } }).filter(Boolean)
            """)
            if len(set(urls)) >= limit * 2:
                break
        await browser.close()
    return list(dict.fromkeys(urls))  # dedupe, keep order


async def download_images(query, output_dir, limit=100, min_side=200):
    os.makedirs(output_dir, exist_ok=True)

    # Hash existing images so reruns don't add duplicates
    hashes = set()
    for f in os.listdir(output_dir):
        if f.lower().endswith(".jpg"):
            with Image.open(os.path.join(output_dir, f)) as im:
                hashes.add(imagehash.phash(im))

    count = len(hashes)
    for url in await collect_urls(query, limit):
        if count >= limit:
            break
        try:
            r = requests.get(url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
            r.raise_for_status()
            img = Image.open(BytesIO(r.content)).convert("RGB")
        except Exception:
            continue
        if min(img.size) < min_side:
            continue
        h = imagehash.phash(img)
        if any(h - other <= 5 for other in hashes):  # near-duplicate
            continue
        hashes.add(h)
        img.save(os.path.join(output_dir, f"{uuid.uuid4().hex[:12]}.jpg"))  # unique names, never overwrite
        count += 1

    print(f"{os.path.basename(output_dir)}: {count} images")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATASET_DIR = "/content/drive/MyDrive/food101_dataset"

CLASSES = {
    "pilaf": "uzbek pilaf plov food",
    "kebab": "kebab shashlik food",
}

for name, query in CLASSES.items():
    await download_images(query, os.path.join(DATASET_DIR, name), limit=100)

Review the folders by hand before training — search results contain off-class images (wraps, drawings, watermarked stock).